In [ ]:
import pandas as pd
import joblib
import os
from google.colab import files
from IPython.display import display

def run_app():
    chemin_modele = 'modele_detection_billets.joblib'
    if not os.path.exists(chemin_modele):
        print(f"❌ Le modèle '{chemin_modele}' est introuvable.")
        return

    print("Chargement du modèle en cours...")
    modele = joblib.load(chemin_modele)

    print("\nVeuillez uploader le fichier CSV à analyser :")
    uploaded = files.upload()

    if not uploaded:
        print("❌ Aucun fichier uploadé.")
        return

    fichier_csv = list(uploaded.keys())[0]

    try:
        df = pd.read_csv(fichier_csv, sep=None, engine='python')
    except Exception as e:
        print(f"❌ Erreur de lecture : {e}")
        return

    colonnes_attendues = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
    if not all(col in df.columns for col in colonnes_attendues):
        print("❌ Format du fichier incorrect.")
        return

    # Arrêt du script si des valeurs manquantes sont détectées
    if df[colonnes_attendues].isnull().values.any():
        print("❌ Erreur : Le fichier contient des valeurs manquantes. Veuillez utiliser un fichier avec des données complètes.")
        return

    print("\nAnalyse géométrique en cours...\n")
    X = df[colonnes_attendues]

    predictions = modele.predict(X)
    probabilites = modele.predict_proba(X)[:, 1]

    df_resultats = df.copy()
    df_resultats['Prediction'] = ["Vrai" if p else "Faux" for p in predictions]
    df_resultats['Confiance (%)'] = probabilites.round(4) * 100

    print("="*40)
    print("        RAPPPORT DE DÉTECTION        ")
    print("="*40)
    print(f"Total analysés : {len(predictions)}")
    print(f"✅ Vrais billets : {sum(predictions)}")
    print(f"🚨 Faux billets  : {len(predictions) - sum(predictions)}")
    print("="*40)

    display(df_resultats[['Prediction', 'Confiance (%)']].head(10))

    fichier_sortie = "resultats_predictions.csv"
    df_resultats.to_csv(fichier_sortie, index=False)
    print(f"\nFichier complet généré : '{fichier_sortie}'")
    files.download(fichier_sortie)

run_app()

Chargement du modèle en cours...

Veuillez uploader le fichier CSV à analyser :


Saving billets_imputes (1).csv to billets_imputes (1).csv

Analyse géométrique en cours...

        RAPPPORT DE DÉTECTION        
Total analysés : 1500
✅ Vrais billets : 1006
🚨 Faux billets  : 494


,Prediction,Confiance (%)
0,Vrai,74.00
1,Vrai,99.98
2,Vrai,99.84
3,Vrai,99.99
4,Vrai,76.55
5,Vrai,98.54
6,Vrai,80.57
7,Vrai,99.92
8,Vrai,99.13
9,Vrai,99.80



Fichier complet généré : 'resultats_predictions.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>